## Loyalty Program Impact
### Goal: 
Compare loyalty members vs non-members in terms of spend and engagement.
### Why it matters: 
Evaluates ROI of the loyalty program.
### How to do it:

- Filter order_items by is_loyalty = true vs false
- Compare per-customer:
- • Average Spend
- • Repeat Orders
- • Lifetime Value

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_fact_order = spark.read.table('global_partner_project.gold.fact_order')
df_dim_user_scd2 = spark.read.table('global_partner_project.gold.dim_user_scd2')

In [0]:
print(df_fact_order.count())
print(df_dim_user_scd2.count())

In [0]:
agg_df_fact_order = df_fact_order.groupBy('user_id','creation_time_utc').agg(F.round(F.sum('item_price'),2).alias('amount'))


In [0]:
df_order_loyalty = (agg_df_fact_order.alias('f').join(
                    df_dim_user_scd2.alias('d'),
                       (F.col('f.user_id') == F.col('d.user_id')) &
                       (F.col('f.creation_time_utc') >= F.col('d.effective_from_ts')) &
                       ( 
                        (F.col('f.creation_time_utc') < F.col('d.effective_to_ts')) | 
                        (F.col('d.effective_to_ts').isNull()) ),
                       'left')
                    .select('f.user_id',
                            'd.is_loyalty',
                            'f.amount',
                            'f.creation_time_utc',
                            'd.effective_from_ts',
                            'd.effective_to_ts')
                    .orderBy('user_id'))

In [0]:
print(df_order_loyalty.count())
df_order_loyalty.display()

In [0]:
agg_df_order_loyalty = (df_order_loyalty.groupBy('user_id','is_loyalty')
    .agg(F.round(F.avg('amount'),2).alias('average_spend'),
        F.round(F.sum('amount'),2).alias('lifetime_value'))
         )

In [0]:
agg_df_order_loyalty.display()

### Repeat Orders

In [0]:
df_dim_item = spark.read.table('global_partner_project.gold.dim_item')

In [0]:
df_item_order = (df_fact_order.alias('f').join(df_dim_item.alias('d'),
                              on='item_id',
                              how='inner')
                .select('f.*',
                        'd.item_name',
                        'd.item_category')
                )

In [0]:
df_order_repeat = (df_item_order.alias('f').join(
                    df_dim_user_scd2.alias('d'),
                       (F.col('f.user_id') == F.col('d.user_id')) &
                       (F.col('f.creation_time_utc') >= F.col('d.effective_from_ts')) &
                       ( 
                        (F.col('f.creation_time_utc') < F.col('d.effective_to_ts')) | 
                        (F.col('d.effective_to_ts').isNull()) ),
                       'left')
                    .select('f.user_id',
                            'd.is_loyalty',
                            'f.item_name',
                            'f.restaurant_id',
                            'f.creation_time_utc',
                            'd.effective_from_ts',
                            'd.effective_to_ts')
                    .orderBy('user_id'))

In [0]:
df_order_repeat.display()

In [0]:
agg_df_rep_order_loyalty = (df_order_repeat.groupBy('user_id','is_loyalty','restaurant_id','item_name')
    .agg(F.count('item_name').alias('order_count')).orderBy('user_id','is_loyalty'))
agg_df_rep_order_loyalty.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.mart.loyalty_repeat_orders;
DROP TABLE IF EXISTS global_partner_project.mart.loyalty_metrics;

In [0]:
agg_df_rep_order_loyalty.write.mode('append').saveAsTable('global_partner_project.mart.loyalty_repeat_orders')
agg_df_order_loyalty.write.mode('append').saveAsTable('global_partner_project.mart.loyalty_metrics')